In [ ]:
from pyspoc import Calculator, Config
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

In [ ]:
project_root = Path('./').resolve().parents[3]
print(project_root)
log_p_value_sum_path = os.path.join(project_root,'mastersResearch_Jun_Sept2025','causalDiscovery_J','ReducedStatistics','log_p_value_sum.py')
print(log_p_value_sum_path)

In [ ]:
log_p_value_sum_config_yaml = f"""
ReducedStatistics:
  {log_p_value_sum_path}:
    logpValueSum:
        schemes:
            scheme_1:
                alpha: 0.05
"""
log_p_value_sum_config_name = "log_p_value_sum"
cfg = Config.from_yaml(log_p_value_sum_config_name, log_p_value_sum_config_yaml)

In [ ]:
base_dir = os.path.join(project_root, 'mastersResearch_Jun_Sept2025', 'causalDiscovery_J')

motivation_dag_1 = os.path.join(base_dir,'motivationAnalysis', 'motivation_dag_1.csv')
motivation_dag_1_df = pd.read_csv(motivation_dag_1)

dag1dataset1 = os.path.join(base_dir, 'dataGen', 'chain100data', 'dag1dataset1.csv')
dag1dataset1_df = pd.read_csv(dag1dataset1)

dag2dataset1 = os.path.join(base_dir, 'dataGen', 'loop100data', 'dag2dataset1.csv')
dag2dataset1_df = pd.read_csv(dag2dataset1)

In [ ]:
calc = Calculator(motivation_dag_1_df)
calc.compute(cfg)
results = calc.results
print(results)

calc = Calculator(dag1dataset1_df)
calc.compute(cfg)
results = calc.results
print(results)

calc = Calculator(dag2dataset1_df)
calc.compute(cfg)
results = calc.results
print(results)

In [ ]:
data_dir1 = os.path.join(base_dir, 'dataGen', 'chain100data')

# Get a list of all items in the directory
all_items = os.listdir(data_dir1)

# Filter for files that are CSVs and create their full paths
csv_files = []
for item in all_items:
    if item.endswith('.csv'):
        file_path = os.path.join(data_dir1, item)
        csv_files.append(file_path)

# Select the first 5 files
files_to_process = csv_files[:5]

# Process each of the selected files
for file_path in files_to_process:
    print(f"Processing file: {os.path.basename(file_path)}")
    
    # Read the CSV file
    df = pd.read_csv(file_path)

    # Initialize and run the Calculator
    calc = Calculator(df)
    calc.compute(cfg)
    results = calc.results
    
    # Print the results
    print(results)
    print("---")

In [ ]:
data_dir2 = os.path.join(base_dir, 'dataGen', 'loop100data')

# Get a list of all items in the directory
all_items = os.listdir(data_dir2)

# Filter for files that are CSVs and create their full paths
csv_files = []
for item in all_items:
    if item.endswith('.csv'):
        file_path = os.path.join(data_dir2, item)
        csv_files.append(file_path)

# Select the first 5 files
files_to_process = csv_files[:5]

# Process each of the selected files
for file_path in files_to_process:
    print(f"Processing file: {os.path.basename(file_path)}")
    
    # Read the CSV file
    df = pd.read_csv(file_path)

    # Initialize and run the Calculator
    calc = Calculator(df)
    calc.compute(cfg)
    results = calc.results
    
    # Print the results
    print(results)
    print("---")

In [ ]:
def vary_N(input_df, N_values, reps=10):
    results_list = []
    for current_N in N_values:
        statistic_sum = 0
        for _ in range(reps):
            subsampled_df = input_df.sample(n=current_N, axis=0).sample(n=100, axis=1)
            calc = Calculator(subsampled_df)
            calc.compute(cfg)
            results = calc.results
            statistic_sum += results.iloc[0, 0]
        avg_statistic = statistic_sum / reps
        results_list.append({
            'N': current_N,
            'avg_statistic': avg_statistic
        })
    return pd.DataFrame(results_list)

N_values = np.arange(100, 1001, 50)
p_value_sum_results=vary_N(dag2dataset1_df, np.arange(100, 1001, 50), 10)

In [ ]:
display(p_value_sum_results)

---
## Experimenting with Polynomial Fits

In [ ]:
plt.figure(figsize=(10, 6))

degree = 2
coefficients = np.polyfit(p_value_sum_results['N'], p_value_sum_results['avg_statistic'], degree)
polynomial = np.poly1d(coefficients)

plt.plot(p_value_sum_results['N'], p_value_sum_results['avg_statistic'], marker='o', linestyle='-',label='Data')

x_fit = np.linspace(p_value_sum_results['N'].min(), p_value_sum_results['N'].max(), 100)
y_fit = polynomial(x_fit)
plt.plot(x_fit, y_fit, color='red', label=f'Polynomial Fit (Degree {degree})')

plt.title('Average Statistic vs. N')
plt.xlabel('N')
plt.ylabel('Average Statistic')
plt.grid(True)
plt.legend()
plt.show()
print(polynomial)

### Attempting to uncouple the statistic and N

In [ ]:
plt.figure(figsize=(10, 6))

degree = 1
coefficients = np.polyfit(p_value_sum_results['N'], (p_value_sum_results['avg_statistic']-1041)/p_value_sum_results['N'], degree)
polynomial = np.poly1d(coefficients)

plt.plot(p_value_sum_results['N'], (p_value_sum_results['avg_statistic']-1041)/p_value_sum_results['N'], marker='o', linestyle='-',label='Data')

x_fit = np.linspace(p_value_sum_results['N'].min(), p_value_sum_results['N'].max(), 100)
y_fit = polynomial(x_fit)
plt.plot(x_fit, y_fit, color='red', label=f'Polynomial Fit (Degree {degree})')

plt.title('Average Statistic vs. N')
plt.xlabel('N')
plt.ylabel('Average Statistic')
plt.grid(True)
plt.legend()
plt.show()
print(polynomial)

In [ ]:
plt.figure(figsize=(10, 6))

degree = 2
coefficients = np.polyfit(p_value_sum_results['N'], (p_value_sum_results['avg_statistic']-9896)/(p_value_sum_results['N']**2), degree)
polynomial = np.poly1d(coefficients)

plt.plot(p_value_sum_results['N'], (p_value_sum_results['avg_statistic']-9896)/(p_value_sum_results['N']**2), marker='o', linestyle='-',label='Data')

x_fit = np.linspace(p_value_sum_results['N'].min(), p_value_sum_results['N'].max(), 100)
y_fit = polynomial(x_fit)
plt.plot(x_fit, y_fit, color='red', label=f'Polynomial Fit (Degree {degree})')

plt.title('Average Statistic vs. N')
plt.xlabel('N')
plt.ylabel('Average Statistic')
plt.grid(True)
plt.legend()
plt.show()
print(polynomial)

fail basically - will try exp next

---
## Experimenting with Exponential Fits

In [ ]:
plt.figure(figsize=(10, 6))

degree = 1
coefficients = np.polyfit(np.log(p_value_sum_results['N']), np.log(p_value_sum_results['avg_statistic']), degree)
polynomial = np.poly1d(coefficients)

plt.scatter(p_value_sum_results['N'], p_value_sum_results['avg_statistic'], marker='o', linestyle='-',label='Data')

x_fit = np.logspace(np.log10(p_value_sum_results['N'].min()), np.log10(p_value_sum_results['N'].max()), 100)
y_fit = np.exp(polynomial(np.log(x_fit)))
plt.plot(x_fit, y_fit, color='red', label=f'Exponential Fit')

plt.title('Average Statistic vs. N')
plt.xlabel('N')
plt.ylabel('Average Statistic')
plt.grid(True)
plt.legend()
plt.show()
print(polynomial)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(p_value_sum_results['N'], p_value_sum_results['avg_statistic'], marker='o', linestyle='-',label='Data')
plt.plot(x_fit, y_fit, color='red', label=f'Exponential Fit')
plt.xscale('log')
plt.yscale('log')
plt.title('Average Statistic vs. N')
plt.xlabel('N')
plt.ylabel('Average Statistic')
plt.grid(True)
plt.legend()
plt.show()
print(polynomial)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(p_value_sum_results['N'], p_value_sum_results['avg_statistic']/(p_value_sum_results['N']**0.05889), marker='o', linestyle='-',label='Data')
plt.xscale('log')
plt.yscale('log')
plt.title('Average Statistic vs. N')
plt.xlabel('N')
plt.ylabel('Average Statistic')
plt.grid(True)
plt.legend()
plt.show()
print(polynomial)

---
### Running with random loop dags in loop100data

In [ ]:
def vary_N_random_from_folder(directory, N_values, reps=10):
    all_files = os.listdir(directory)
    dataset_files = [f for f in all_files if f.startswith('dag2dataset') and f.endswith('.csv')]
    random_file = random.choice(dataset_files)
    return vary_N(pd.read_csv(os.path.join(directory, random_file)), N_values, reps)

In [ ]:
loop_data_dir = os.path.join(base_dir, 'dataGen', 'loop100data')
p_value_sum_results2=vary_N_random_from_folder(loop_data_dir, N_values, reps=10)

In [ ]:
display(p_value_sum_results2)

In [ ]:
plt.figure(figsize=(10, 6))

degree = 2
coefficients = np.polyfit(p_value_sum_results2['N'], p_value_sum_results2['avg_statistic'], degree)
polynomial = np.poly1d(coefficients)

plt.plot(p_value_sum_results2['N'], p_value_sum_results2['avg_statistic'], marker='o', linestyle='-',label='Data')

x_fit = np.linspace(p_value_sum_results2['N'].min(), p_value_sum_results2['N'].max(), 100)
y_fit = polynomial(x_fit)
plt.plot(x_fit, y_fit, color='red', label=f'Polynomial Fit (Degree {degree})')

plt.title('Average Statistic vs. N')
plt.xlabel('N')
plt.ylabel('Average Statistic')
plt.grid(True)
plt.legend()
plt.show()
print(polynomial)

In [ ]:
plt.figure(figsize=(10, 6))

degree = 1
coefficients = np.polyfit(np.log(p_value_sum_results2['N']), np.log(p_value_sum_results2['avg_statistic']), degree)
polynomial = np.poly1d(coefficients)

plt.scatter(p_value_sum_results2['N'], p_value_sum_results2['avg_statistic'], marker='o', linestyle='-',label='Data')

x_fit = np.logspace(np.log10(p_value_sum_results2['N'].min()), np.log10(p_value_sum_results2['N'].max()), 100)
y_fit = np.exp(polynomial(np.log(x_fit)))
plt.plot(x_fit, y_fit, color='red', label=f'Exponential Fit')

plt.title('Average Statistic vs. N')
plt.xlabel('N')
plt.ylabel('Average Statistic')
plt.grid(True)
plt.legend()
plt.show()
print(polynomial)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(p_value_sum_results2['N'], p_value_sum_results2['avg_statistic'], marker='o', linestyle='-',label='Data')
plt.plot(x_fit, y_fit, color='red', label=f'Exponential Fit')
plt.xscale('log')
plt.yscale('log')
plt.title('Average Statistic vs. N')
plt.xlabel('N')
plt.ylabel('Average Statistic')
plt.grid(True)
plt.legend()
plt.show()
print(polynomial)